# Can You Run Isaac Sim on a Free Kaggle GPU?

**Four walls, four fixes, and an honest answer. Everything here was actually run on Kaggle — the errors are real, captured output, not hypotheticals.**

---

> Part of an open series on running **NVIDIA Isaac Sim on free GPUs**.
> Isaac Sim is free software (Apache 2.0) — only compute ever costs money,
> and this series is about not paying for that either.
>
> If this is useful, an upvote helps other people find it. Questions in the
> comments get answered.

---

## The question

NVIDIA **Isaac Sim** is the industry-standard robotics simulator — GPU physics,
RTX rendering, synthetic data generation, RL environments via Isaac Lab. It is
free software (Apache 2.0).

The hardware usually is not. Almost every guide opens with "get an RTX
workstation" or "spin up a g6e instance." Kaggle gives away **30 GPU hours a
week on 2x T4** to anyone with a verified phone number.

So: can you run Isaac Sim on that?

**Short answer: no — and the reasons are specific, diagnosable, and worth
knowing.** I hit four separate walls. Three of them I got past. The fourth is
where it ends.

This notebook is the teardown. Every error below is real captured output from
running this on Kaggle, not a hypothetical. If you are about to spend an
evening on this, I hope it saves you the evening.

**The cells here run safely and stop before the crash.** Reproducing the final
failure kills the kernel outright, so that part is documented rather than
executed.

## Wall 0 — Preflight

Four things can independently break this, and each takes ~10 minutes of
install to discover the slow way. Check them in five seconds instead.

In [ ]:
import os, platform, shutil, subprocess, sys

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True,
                              text=True, timeout=60).stdout.strip()
    except Exception:
        return ""

print("=" * 62)

gpu = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version "
         "--format=csv,noheader")
for line in (gpu.splitlines() or ["(no GPU)"]):
    print(" GPU:", line)

glibc = platform.libc_ver()[1]
print(f" GLIBC: {glibc}  (Isaac Sim needs >= 2.34)")
print(f" Python: {sys.version.split()[0]}")

for path in ["/kaggle/working", "/kaggle/temp", "/tmp"]:
    if os.path.isdir(path):
        print(f" disk: {shutil.disk_usage(path).free/1e9:7.1f} GB free at {path}")
print("=" * 62)

Note the disk numbers. On the run that produced this notebook,
`/kaggle/working` had **20.9 GB** free while `/tmp` had **1,102 GB**. The
`isaacsim` wheels are roughly **20 GB**. Installing into the working directory
is marginal at best — point large caches at `/tmp`.

### The RT core trap

Worth stopping on, because it is the most counterintuitive fact in cloud
robotics simulation.

Isaac Sim's renderer needs **RT cores** — dedicated ray-tracing hardware.

| GPU | Compute cap | RT cores? |
|---|---|---|
| T4 (Kaggle/Colab free) | 7.5 | ✅ Yes |
| L4 / L40S | 8.9 | ✅ Yes |
| RTX 3090 / 4090 | 8.6 / 8.9 | ✅ Yes |
| **A100** | **8.0** | ❌ **No** |
| **H100** | **9.0** | ❌ **No** |

The expensive datacenter parts strip RT cores out — their customers train
transformers, they do not render. So a $2/hr A100 is **worse for this job**
than Kaggle's free T4.

Caveat worth knowing: this only bites when you *render*. For physics-only RL
on state vectors, an A100 is fine.

## Wall 1 — The Python version pin

The first failure, 40 seconds in:

```
ERROR: Ignored the following versions that require a different python version:
       4.5.0.0 Requires-Python ==3.10.*; 5.0.0.0 Requires-Python ==3.11.*
ERROR: Could not find a version that satisfies the requirement isaacsim==4.5.0
       (from versions: 6.0.0.0, 6.0.0.1, 6.0.1.0)
```

**The `isaacsim` wheels are pinned to exact Python versions:**

| Isaac Sim | Requires Python | Works on Kaggle (3.12)? |
|---|---|---|
| 4.0 – 4.5 | **3.10 only** | ❌ |
| 5.0 – 5.1 | **3.11 only** | ❌ |
| **6.0.x** | **3.12** | ✅ |

Nearly every tutorial online pins `==4.5.0`, because that was current when it
was written. On Python 3.12 that install *cannot* succeed, and pip's error
never mentions Python — the `from versions:` list is the real signal.

**Fix:** detect the interpreter and pick the matching build.

In [ ]:
py = sys.version_info
ISAAC_FOR_PY = {(3, 10): "4.5.0", (3, 11): "5.1.0", (3, 12): "6.0.1.0"}
version = ISAAC_FOR_PY.get((py.major, py.minor))

print(f"Python {py.major}.{py.minor} -> isaacsim=={version}"
      if version else f"no known isaacsim build for Python {py.major}.{py.minor}")
print("\nlist real options with:")
print("  pip index versions isaacsim --extra-index-url https://pypi.nvidia.com")

## Wall 2 — Vulkan has no driver registered

With the right version, the install succeeds — **9 minutes 34 seconds**, about
20 GB. Then `SimulationApp` starts, and:

```
[Error] [omni.rtx] VkResult: ERROR_INCOMPATIBLE_DRIVER
[Error] [omni.rtx] vkCreateInstance failed. Vulkan 1.1 is not supported,
                   or your driver requires an update
```

...followed by the kernel dying outright. `DeadKernelError`, no traceback.

**The message blames the driver. The driver is fine.** The T4 has RT cores and
a working NVIDIA driver (580.159.04). What Kaggle's container lacks is
`/usr/share/vulkan/icd.d/nvidia_icd.json` — the small JSON that tells the
Vulkan loader which library implements the driver. Isaac Sim renders through
Vulkan, so without that registration it cannot start.

**Fix:** install the loader and write the ICD file yourself. This one works —
run it and see.

In [ ]:
import glob, json

print(sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader"))

sh("apt-get update -qq && apt-get install -y -qq libvulkan1 vulkan-tools")

cands = (glob.glob("/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so*")
         + glob.glob("/usr/local/nvidia/lib64/libGLX_nvidia.so*")
         + glob.glob("/usr/lib/libGLX_nvidia.so*"))
print("libGLX_nvidia found:", cands or "NONE")

os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
icd = {"file_format_version": "1.0.0",
       "ICD": {"library_path": cands[0] if cands else "libGLX_nvidia.so.0",
               "api_version": "1.3.242"}}
with open("/usr/share/vulkan/icd.d/nvidia_icd.json", "w") as f:
    json.dump(icd, f, indent=2)
print("wrote nvidia_icd.json ->", icd["ICD"]["library_path"])

info = sh("vulkaninfo --summary")
print("\nVulkan sees a device:", "deviceName" in info or "GPU id" in info)
print(info[:600] or "(no output — vulkaninfo unavailable)")

If that printed **`Vulkan sees a device: True`**, wall 2 is down. On the run
behind this notebook it did, and `ERROR_INCOMPATIBLE_DRIVER` disappeared.

That is a genuinely useful fix well beyond Isaac Sim — it applies to any
Vulkan workload in a container that ships CUDA but not the graphics ICD.

## Wall 3 — NumPy ABI mismatch

Past Vulkan, Kit boots through ~16,000 lines of extension startup, and then:

```
[Error] [omni.ext._impl.custom_importer] Failed to import python module
        isaacsim.core.prims. Error: cannot import name '_center'
        from 'numpy._core.umath'

[Error] [omni.ext._impl.custom_importer] Failed to import python module
        isaacsim.core.api. Error: cannot import name 'SingleGeometryPrim'
```

Isaac Sim 6.0.1.0 is built against an older NumPy than Kaggle ships
(**2.3.1**). `numpy._core.umath._center` is a private symbol that moved. The
whole `isaacsim.core` tree fails to import.

**Fix in principle:** `pip install "numpy<2.1"` before importing. In practice
this fights Kaggle's preinstalled stack — the image already warns
`numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.1`, so the
conflict predates Isaac Sim and downgrading ripples outward.

## Wall 4 — PhysX cannot get a CUDA context

This is where it ends.

```
[Error] [omni.physx.foundation.plugin] Failed to create Cuda Context Manager.
[Error] [omni.physx.plugin] Unable to create PxCudaContextManager!
[Warning] [omni.physx.plugin] PhysX warning: Minimum GPU compute capability
                              7.0 is required
[49.087s] Simulation App Starting
Kernel died while waiting for execute reply.
```

A T4 is compute capability **7.5**. It clears the stated 7.0 bar comfortably.
PhysX simply cannot create a CUDA context inside this container.

I tried pinning to a single GPU — Kaggle allocates 2x T4, and device selection
is a known failure point:

```python
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.argv += ["--/physics/cudaDevice=0"]
```

**It did not help.** Identical error, identical death.

Unlike walls 1–3, this one is not a missing config file or a version pin. It
looks like the container simply does not expose the CUDA driver interface Kit
needs, and that is not fixable from inside a notebook.

## The verdict

**Isaac Sim does not run on Kaggle.** Not "runs slowly" — does not start.

Here is how far you get, which is further than you would guess:

| Stage | Result |
|---|---|
| `pip install isaacsim==6.0.1.0` | ✅ works (9m34s, ~20 GB) |
| Vulkan driver registration | ✅ fixable (write the ICD yourself) |
| `isaacsim.core` imports | ❌ NumPy ABI mismatch |
| PhysX CUDA context | ❌ **hard stop** |

### What to do instead

The pattern that actually works is to **split generation from publication**:

> **Generate on a persistent GPU box → publish the artifacts to Kaggle.**

Do the Isaac Sim work somewhere with a disk that survives and a container
built for it. **Lightning AI's free tier** gives 80 GPU hours a month *with
persistent storage*, so the 20 GB install happens once instead of every
session. Upload the outputs as a Kaggle Dataset. Kaggle notebooks then consume
pre-generated data and need a GPU only for light work.

Free options ranked for this specific job:

| Method | Free allowance | Isaac Sim? |
|---|---|---|
| **Lightning AI** | 80 GPU hr/mo, persistent, sudo | ✅ best free option |
| **NVIDIA DLI courses** | hosted labs | ✅ real Isaac Sim, zero setup |
| Google Colab | best-effort T4 | 🟡 same container problems |
| **Kaggle** | 30 GPU hr/wk, 2x T4 | ❌ this notebook |

Kaggle is still excellent for the *rest* of the robotics-simulation stack.
**MuJoCo and MJX run here perfectly** — see the companion notebook in this
series, which measures integrator accuracy and parallel scaling entirely on
Kaggle hardware.

### The three transferable lessons

1. **RT cores, not FLOPs.** A100/H100 have no RT cores. The cheap GPU is often
   the right one.
2. **`isaacsim` wheels are pinned to exact Python versions.** Every tutorial
   pinning `==4.5.0` is broken on Python 3.12, and pip will not tell you why.
3. **`ERROR_INCOMPATIBLE_DRIVER` usually means a missing ICD file**, not a bad
   driver — a 4-line JSON fixes it.

If you get Isaac Sim running on Kaggle, I would genuinely like to know how —
**post it in the comments** and I will update this notebook and credit you.
Negative results are worth publishing, but they are worth correcting more.

---

## Reproducing this

Every notebook in this series runs on free infrastructure. Nothing here needs
a paid GPU.

| Method | Free allowance | Best for |
|---|---|---|
| Kaggle | 30 GPU hr/week, 2x T4 | Running this notebook as-is |
| Lightning AI | 80 GPU hr/month, persistent disk | Heavy Isaac Sim generation |
| Google Colab | best-effort T4 | Quick smoke tests |
| NVIDIA DLI | free hosted labs | Learning the Isaac Sim GUI |

**The one gotcha worth remembering:** Isaac Sim needs **RT cores**. A T4, L4,
L40S or any RTX card is fine. An **A100 or H100 is not** — those have no RT
cores, so the RTX renderer is unsupported or unusably slow. It is the most
counterintuitive constraint in cloud robotics simulation, and it bites people
who assume the more expensive GPU must be the better one.

*Series index and full source: see the linked dataset description.*